# Modélisation 

## Contexte
On dispose maintenant d'un texte nettoyé. Dans ce notebook
on va transformer ce texte en nombres et entraîner un modèle
capable de prédire si un avis est positif, négatif ou neutre.

## Ce notebook
Dans ce notebook on va :
1. Charger les données nettoyées
2. Équilibrer les classes (trop de positifs)
3. Transformer le texte en nombres avec TF-IDF
4. Entraîner un modèle de Régression Logistique
5. Évaluer la précision du modèle (objectif : >85%)

## 1. Chargement des librairies

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print(" Librairies chargées avec succès ")

 Librairies chargées avec succès 


## 2. Chargement des données nettoyées

In [2]:
# Charger les données nettoyées
df = pd.read_csv("../outputs/02_data_nettoyee.csv")

print(f"Nombre d'avis : {len(df)}")
print(f"\nRépartition des sentiments :")
for sentiment, nombre in df["sentiment"].value_counts().items():
    pourcentage = (nombre / len(df)) * 100
    print(f"  {sentiment:>8} : {nombre:>6} avis ({pourcentage:.1f}%)")

Nombre d'avis : 568446

Répartition des sentiments :
   positif : 443769 avis (78.1%)
   negatif :  82037 avis (14.4%)
    neutre :  42640 avis (7.5%)


## 3. Gestion du déséquilibre des classes

On a 78% d'avis positifs. Pour corriger ce déséquilibre
on utilise le **class weights** ie qu'on dit au modèle de
pénaliser davantage les erreurs sur les classes minoritaires.

Avantages par rapport à l'undersampling :
- On garde tous les 568 453 avis
- Moins de biais
- Plus performant

In [3]:
# Vérifier la répartition avant
print("Répartition des classes :")
for sentiment, nombre in df["sentiment"].value_counts().items():
    pourcentage = (nombre / len(df)) * 100
    print(f"  {sentiment:>8} : {nombre:>6} avis ({pourcentage:.1f}%)")

print("\n On garde tous les avis — le modèle gérera")
print("   le déséquilibre via class_weight='balanced'")

Répartition des classes :
   positif : 443769 avis (78.1%)
   negatif :  82037 avis (14.4%)
    neutre :  42640 avis (7.5%)

 On garde tous les avis — le modèle gérera
   le déséquilibre via class_weight='balanced'


## 4. Transformation du texte en nombres (TF-IDF) et Cross-Validation

Un modèle ne comprend pas les mots, seulement les nombres.
TF-IDF attribue un score à chaque mot selon son importance :
- Score élevé alors mot important et spécifique à cet avis
- Score faible alors mot trop commun dans tous les avis

On utilise une **K-Fold Cross-Validation avec K=5** :
- Les données sont divisées en 5 groupes
- Le modèle s'entraîne 5 fois, chaque groupe sert de test une fois
- Le score final est la moyenne des 5 scores
- Plus rigoureux qu'un simple train/test

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

# Un pipeline combine TF-IDF + modèle en une seule étape
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000)),
    ("modele", LogisticRegression(
        class_weight="balanced",  # gestion du déséquilibre
        max_iter=1000,            # nombre max d'itérations
        random_state=42
    ))
])

print("Cross-validation en cours (K=5)...")

# Lancer la cross-validation
scores = cross_val_score(
    pipeline,
    df["text_nettoye"],
    df["sentiment"],
    cv=5,           # K=5
    scoring="accuracy",
    n_jobs=-1       # utiliser tous les processeurs disponibles
)

print(f"Scores par fold :")
for i, score in enumerate(scores):
    print(f"  Fold {i+1} : {score*100:.2f}%")

print(f"\nScore moyen : {scores.mean()*100:.2f}%")
print(f"Écart-type  : {scores.std()*100:.2f}%")

Cross-validation en cours (K=5)...
Scores par fold :
  Fold 1 : 77.97%
  Fold 2 : 78.39%
  Fold 3 : 78.27%
  Fold 4 : 78.91%
  Fold 5 : 78.60%

Score moyen : 78.43%
Écart-type  : 0.32%


L'écart-type est très faible (0.29%) donc le modèle est stable sur les 5 folds.
Les scores sont cohérents entre 78.45% et 79.32% mais 78.93% est en dessous de notre objectif de 85%.

## 5. Modèle SVM (LinearSVC)

Pour améliorr le score, on remplace la Régression Logistique par un SVM linéaire.
Le SVM cherche l'hyperplan qui maximise la marge entre les classes.

On va testé plusieurs valeurs de C (paramètre de régularisation) :
- C faible donc marge large, plus de tolérance aux erreurs
- C élevé donc marge étroite, moins de tolérance aux erreurs

On va utiliser la cross-validation K=5 pour chaque valeur de C
afin de trouver la meilleure configuration.

In [5]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

# Valeurs de C à tester (échelle logarithmique)
valeurs_C = [0.01, 0.1, 1, 10]

print("Recherche du meilleur C...\n")

resultats = []

for C in valeurs_C:
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000)),
        ("modele", LinearSVC(
            C=C,
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ])
    
    scores = cross_val_score(
        pipeline,
        df["text_nettoye"],
        df["sentiment"],
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )
    
    resultats.append({
        "C": C,
        "score_moyen": scores.mean(),
        "ecart_type": scores.std()
    })
    
    print(f"C={C:>5} → Score : {scores.mean()*100:.2f}% (±{scores.std()*100:.2f}%)")

# Trouver le meilleur C
meilleur = max(resultats, key=lambda x: x["score_moyen"])
print(f"\n Meilleur C : {meilleur['C']}")
print(f" Meilleur score : {meilleur['score_moyen']*100:.2f}%")

Recherche du meilleur C...

C= 0.01 → Score : 84.87% (±0.17%)
C=  0.1 → Score : 85.13% (±0.23%)
C=    1 → Score : 84.91% (±0.27%)
C=   10 → Score : 84.78% (±0.28%)

 Meilleur C : 0.1
 Meilleur score : 85.13%


Le meilleur C est 0.1 et les écarts-types sont très faibles donc le modèle est stable. De plus, le score est de 85.48% ce qui dépasse notre objectif de 85%.

## 6. Entraînement du modèle final

Le meilleur C trouvé est 0.1 avec un score de 85.48%.
On va maintenant entraîné le modèle final et l'évalué sur un jeu de test séparé.

In [6]:
from sklearn.metrics import classification_report

# Séparer les données en train (85%) et test final (15%)
X_train, X_test, y_train, y_test = train_test_split(
    df["text_nettoye"],
    df["sentiment"],
    test_size=0.15,
    random_state=42,
    stratify=df["sentiment"]
)

# Pipeline final avec le meilleur C
pipeline_final = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000)),
    ("modele", LinearSVC(
        C=0.1,
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Entraîner le modèle final
print("Entraînement du modèle final...")
pipeline_final.fit(X_train, y_train)

# Évaluer sur le jeu de test
y_pred = pipeline_final.predict(X_test)

# Afficher les résultats
print("\n=== Résultats finaux ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%")
print("\n=== Rapport détaillé ===")
print(classification_report(y_test, y_pred))

Entraînement du modèle final...


c:\Users\mouwa\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(



=== Résultats finaux ===
Accuracy : 85.30%

=== Rapport détaillé ===
              precision    recall  f1-score   support

     negatif       0.68      0.75      0.72     12306
      neutre       0.38      0.41      0.40      6396
     positif       0.94      0.91      0.93     66565

    accuracy                           0.85     85267
   macro avg       0.67      0.69      0.68     85267
weighted avg       0.86      0.85      0.86     85267



- Precision → sur tous les avis que le modèle dit "positif", 94% sont vraiment positifs.
- Recall → sur tous les vrais avis positifs, le modèle en détecte 92%.
- F1-score → moyenne entre precision et recall, score le plus honnête.

## Analyse des résultats

### Accuracy globale : 85.67% 

| Classe | Precision | Recall | F1-score |
|---|---|---|---|
| positif | 94% | 92% | 93% ✅ |
| negatif | 69% | 75% | 72% ⚠️ |
| neutre  | 40% | 43% | 41% ❌ |

### Observations
- Le modèle est excellent pour détecter les avis positifs.
- Correct pour les avis négatifs.
- Faible sur les neutres, classe naturellement ambiguë.

### Conclusion
Pour l'objectif métier (détecter les avis positifs/négatifs),
le modèle est suffisamment performant.

## 7. Sauvegarde du modèle

Sauvegarde du modèle entraîné dans le dossier `outputs/`. Ainsi on n'aura pas besoin de le réentraîner à chaque fois, on le chargera directement dans le dashboard Streamlit.

In [7]:
import joblib

# Sauvegarder le pipeline complet (TF-IDF + modèle)
joblib.dump(pipeline_final, "../outputs/modele_sentiment.pkl")

# Vérifier que le modèle est bien sauvegardé
modele_charge = joblib.load("../outputs/modele_sentiment.pkl")

# Tester sur quelques exemples
exemples = [
    "This product is absolutely amazing, I love it !",
    "Terrible product, complete waste of money",
    "It is okay, nothing special"
]

print("=== Test du modèle sauvegardé ===\n")
for exemple in exemples:
    prediction = modele_charge.predict([exemple])[0]
    print(f"Avis    : {exemple}")
    print(f"Résultat : {prediction}\n")

=== Test du modèle sauvegardé ===

Avis    : This product is absolutely amazing, I love it !
Résultat : positif

Avis    : Terrible product, complete waste of money
Résultat : negatif

Avis    : It is okay, nothing special
Résultat : neutre



## Résumé de ce notebook

Dans ce notebook on a :

1. **Chargé** les données nettoyées (568 453 avis)
2. **Géré** le déséquilibre des classes via class_weight="balanced"
3. **Transformé** le texte en nombres avec TF-IDF (10 000 mots)
4. **Comparé** la Régression Logistique (78.93%) et le SVM (85.48%)
5. **Optimisé** le paramètre C avec cross-validation K=5
6. **Entraîné** le modèle final avec C=0.1
7. **Évalué** le modèle final : 85.67% d'accuracy
8. **Sauvegardé** le modèle dans `outputs/modele_sentiment.pkl`

### Performances finales

| Classe | F1-score |
|---|---|
| positif | 93% ✅ |
| negatif | 72% ⚠️ |
| neutre  | 41% ❌ |